<a href="https://colab.research.google.com/github/prrtk/viscometer-app/blob/main/mtp2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Physics-Informed Neural Network for Microfluidic Droplet Dynamics
===============================================================

This implementation mirrors the methodology from the thesis:
"Physics-Informed Machine Learning for Microfluidic Droplet Dynamics: A Unified Computational Framework"

Components:
1. High-fidelity physics-based data generation (Navier-Stokes + Level Set)
2. Academic-grade PINN architecture with adaptive loss weighting
3. Professional validation suite with quantitative metrics

Author: Pratik Choudhury (20CH30065)
Advisor: Prof. Arnab Atta
Institution: Indian Institute of Technology Kharagpur
"""

import os
import time
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.axes_grid1 import make_axes_locatable
from typing import Tuple, Dict, List, Optional, Callable
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# =============================================================================
# Part 1: High-Fidelity Physics-Based Data Generation
# =============================================================================

@dataclass
class SimulationParameters:
    """Parameters for microfluidic T-junction simulation"""
    # Geometry parameters
    main_channel_width: float = 100e-6  # meters
    side_channel_width: float = 50e-6    # meters
    channel_length: float = 500e-6       # meters

    # Fluid properties
    rho_continuous: float = 1000.0       # kg/m^3 (water)
    rho_dispersed: float = 900.0         # kg/m^3 (oil)
    mu_continuous: float = 0.001         # Pa.s (water)
    mu_dispersed_range: Tuple[float, float] = (0.001, 0.005)  # Pa.s (oil)
    surface_tension: float = 0.045       # N/m

    # Simulation parameters
    simulation_time: float = 2.5         # seconds
    max_timestep: float = 0.005          # seconds
    mesh_size: int = 1171                # number of elements

    # Boundary conditions
    inlet_velocity_range: Tuple[float, float] = (0.0026, 0.0046)  # m/s

class MicrofluidicSimulator:
    """
    Numerical solver for microfluidic T-junction droplet formation using
    Navier-Stokes equations coupled with Level Set method.

    This implements a simplified 2D finite difference solver that captures
    the essential physics while being computationally tractable for generating
    training data.
    """

    def __init__(self, params: SimulationParameters):
        self.params = params
        self._setup_domain()
        self._initialize_fields()

    def _setup_domain(self):
        """Initialize the computational domain and grid"""
        self.nx = 100  # points in x-direction
        self.ny = 50   # points in y-direction
        self.dx = self.params.channel_length / (self.nx - 1)
        self.dy = self.params.main_channel_width / (self.ny - 1)

        # Create grid
        self.x = np.linspace(0, self.params.channel_length, self.nx)
        self.y = np.linspace(0, self.params.main_channel_width, self.ny)
        self.X, self.Y = np.meshgrid(self.x, self.y, indexing='ij')

        # Identify channel regions
        self._identify_geometry()

    def _identify_geometry(self):
        """Mark different regions of the T-junction"""
        # Main channel (horizontal)
        self.main_channel = np.ones((self.nx, self.ny), dtype=bool)

        # Side channel (vertical)
        side_channel_start = int(0.4 * self.nx)
        side_channel_end = int(0.6 * self.nx)
        side_width = int(self.params.side_channel_width / self.dy)

        self.side_channel = np.zeros((self.nx, self.ny), dtype=bool)
        self.side_channel[side_channel_start:side_channel_end, :side_width] = True

        # Combined domain
        self.domain = self.main_channel | self.side_channel

    def _initialize_fields(self):
        """Initialize velocity, pressure, and level set fields"""
        # Velocity fields (u, v components)
        self.u = np.zeros((self.nx, self.ny))
        self.v = np.zeros((self.nx, self.ny))

        # Pressure field
        self.p = np.zeros((self.nx, self.ny))

        # Level set function (φ > 0: continuous phase, φ < 0: dispersed phase)
        self.phi = np.ones((self.nx, self.ny))

        # Initialize a small plug of dispersed phase in the side channel
        plug_start = int(0.45 * self.nx)
        plug_end = int(0.55 * self.nx)
        plug_width = int(0.3 * self.params.side_channel_width / self.dy)
        self.phi[plug_start:plug_end, :plug_width] = -1

        # Density and viscosity fields
        self.rho = np.where(self.phi > 0,
                           self.params.rho_continuous,
                           self.params.rho_dispersed)
        self.mu = np.where(self.phi > 0,
                          self.params.mu_continuous,
                          self.params.mu_dispersed_range[0])

    def _compute_surface_tension_force(self):
        """Calculate surface tension force using level set"""
        # Compute gradient of level set function
        grad_phi_x, grad_phi_y = np.gradient(self.phi, self.dx, self.dy)
        grad_phi_mag = np.sqrt(grad_phi_x**2 + grad_phi_y**2)

        # Compute curvature
        with np.errstate(divide='ignore', invalid='ignore'):
            kappa = np.zeros_like(self.phi)
            mask = grad_phi_mag > 1e-10
            kappa[mask] = (grad_phi_x[mask]**2 * np.gradient(grad_phi_y, axis=0)[mask] -
                          2 * grad_phi_x[mask] * grad_phi_y[mask] * np.gradient(grad_phi_x, axis=1)[mask] +
                          grad_phi_y[mask]**2 * np.gradient(grad_phi_x, axis=0)[mask]) / (grad_phi_mag[mask]**3)

        # Compute surface tension force
        F_st = np.zeros((2, self.nx, self.ny))
        F_st[0] = self.params.surface_tension * kappa * grad_phi_x
        F_st[1] = self.params.surface_tension * kappa * grad_phi_y

        return F_st

    def _advect_level_set(self, dt: float):
        """Advect the level set function using the velocity field"""
        # Compute gradients
        grad_phi_x, grad_phi_y = np.gradient(self.phi, self.dx, self.dy)

        # Update level set function
        self.phi -= dt * (self.u * grad_phi_x + self.v * grad_phi_y)

        # Reinitialize to maintain signed distance property (simplified)
        self._reinitialize_level_set()

    def _reinitialize_level_set(self):
        """Simplified reinitialization of level set function"""
        # In a full implementation, we would use a proper reinitialization scheme
        # Here we just ensure the interface remains sharp
        self.phi = np.tanh(self.phi / (0.5 * self.dx))

    def _solve_navier_stokes(self, dt: float):
        """Solve incompressible Navier-Stokes equations using projection method"""
        # Compute intermediate velocity (ignoring pressure)
        u_star = self.u.copy()
        v_star = self.v.copy()

        # Compute convective terms
        conv_u = self.u * np.gradient(self.u, self.dx, axis=0) + self.v * np.gradient(self.u, self.dy, axis=1)
        conv_v = self.u * np.gradient(self.v, self.dx, axis=0) + self.v * np.gradient(self.v, self.dy, axis=1)

        # Compute viscous terms
        viscous_u = self.mu * (np.gradient(np.gradient(self.u, self.dx, axis=0), self.dx, axis=0) +
                                  np.gradient(np.gradient(self.u, self.dy, axis=1), self.dy, axis=1)) / self.rho
        viscous_v = self.mu * (np.gradient(np.gradient(self.v, self.dx, axis=0), self.dx, axis=0) +
                                  np.gradient(np.gradient(self.v, self.dy, axis=1), self.dy, axis=1)) / self.rho

        # Compute surface tension force
        F_st = self._compute_surface_tension_force()

        # Update intermediate velocity
        u_star += dt * (-conv_u + viscous_u + F_st[0]/self.rho)
        v_star += dt * (-conv_v + viscous_v + F_st[1]/self.rho)

        # Solve pressure Poisson equation (simplified)
        div_u_star = np.gradient(u_star, self.dx, axis=0) + np.gradient(v_star, self.dy, axis=1)
        self.p = self._solve_poisson(div_u_star / dt)

        # Correct velocity to ensure incompressibility
        grad_p_x, grad_p_y = np.gradient(self.p, self.dx, self.dy)
        self.u = u_star - dt * grad_p_x / self.rho
        self.v = v_star - dt * grad_p_y / self.rho

    def _solve_poisson(self, rhs: np.ndarray, max_iter: int = 1000, tol: float = 1e-6) -> np.ndarray:
        """Solve Poisson equation using Jacobi iteration"""
        p = np.zeros_like(rhs)
        p_new = p.copy()

        for _ in range(max_iter):
            p_new[1:-1, 1:-1] = 0.25 * (p[2:, 1:-1] + p[:-2, 1:-1] +
                                         p[1:-1, 2:] + p[1:-1, :-2] -
                                         rhs[1:-1, 1:-1] * self.dx**2)

            # Apply boundary conditions (Neumann)
            p_new[0, :] = p_new[1, :]  # Left boundary
            p_new[-1, :] = p_new[-2, :]  # Right boundary
            p_new[:, 0] = p_new[:, 1]  # Bottom boundary
            p_new[:, -1] = p_new[:, -2]  # Top boundary

            # Check convergence
            residual = np.max(np.abs(p_new - p))
            if residual < tol:
                break

            p = p_new.copy()

        return p

    def run_simulation(self, u_in: float, mu_dispersed: float,
                       output_interval: int = 10) -> Dict[str, np.ndarray]:
        """
        Run the simulation with given parameters

        Args:
            u_in: Inlet velocity of dispersed phase (m/s)
            mu_dispersed: Viscosity of dispersed phase (Pa.s)
            output_interval: Number of timesteps between outputs

        Returns:
            Dictionary containing time series of simulation results
        """
        # Set parameters
        self.mu_dispersed = mu_dispersed
        self.u_in = u_in

        # Initialize fields
        self._initialize_fields()

        # Apply boundary conditions
        self._apply_boundary_conditions()

        # Determine timestep based on CFL condition
        dt = self._compute_stable_timestep()
        num_steps = int(self.params.simulation_time / dt)

        # Prepare output storage
        output_times = []
        output_u = []
        output_v = []
        output_p = []
        output_phi = []

        # Main simulation loop
        for step in range(num_steps):
            # Update viscosity field based on current phase
            self.mu = np.where(self.phi > 0,
                              self.params.mu_continuous,
                              self.mu_dispersed)

            # Solve Navier-Stokes equations
            self._solve_navier_stokes(dt)

            # Advect level set function
            self._advect_level_set(dt)

            # Apply boundary conditions
            self._apply_boundary_conditions()

            # Store output at specified intervals
            if step % output_interval == 0:
                output_times.append(step * dt)
                output_u.append(self.u.copy())
                output_v.append(self.v.copy())
                output_p.append(self.p.copy())
                output_phi.append(self.phi.copy())

                # Print progress
                if step % (10 * output_interval) == 0:
                    print(f"Step {step}/{num_steps} (t = {step*dt:.3f}s)")

        # Convert outputs to numpy arrays
        results = {
            'times': np.array(output_times),
            'u': np.stack(output_u),
            'v': np.stack(output_v),
            'p': np.stack(output_p),
            'phi': np.stack(output_phi),
            'x': self.X,
            'y': self.Y,
            'params': {
                'u_in': u_in,
                'mu_dispersed': mu_dispersed,
                'dt': dt,
                'dx': self.dx,
                'dy': self.dy
            }
        }

        return results

    def _apply_boundary_conditions(self):
        """Apply boundary conditions to velocity fields"""
        # Inlet (side channel) - dispersed phase
        side_inlet = self.side_channel & (self.X <= self.dx)
        self.u[side_inlet] = self.u_in
        self.v[side_inlet] = 0

        # Inlet (main channel) - continuous phase
        main_inlet = self.main_channel & (self.Y <= self.dy)
        self.u[main_inlet] = 1.5 * self.u_in  # Higher velocity for continuous phase
        self.v[main_inlet] = 0

        # Outlet (pressure outlet)
        outlet = self.main_channel & (self.X >= self.params.channel_length - self.dx)
        self.p[outlet] = 0  # Gauge pressure
        # Neumann condition for velocity
        self.u[outlet] = self.u[outlet - 1]
        self.v[outlet] = self.v[outlet - 1]

        # Walls (no-slip)
        walls = ~self.domain
        self.u[walls] = 0
        self.v[walls] = 0

    def _compute_stable_timestep(self) -> float:
        """Compute stable timestep based on CFL condition"""
        max_velocity = max(self.u_in, 1.5 * self.u_in)
        dt_cfl = 0.25 * min(self.dx, self.dy) / max_velocity

        # Additional constraint from surface tension
        dt_cap = np.sqrt((self.rho_continuous + self.rho_dispersed) *
                        min(self.dx, self.dy)**3 / (2 * np.pi * self.params.surface_tension))

        return min(dt_cfl, dt_cap, self.params.max_timestep)

    def save_to_hdf5(self, results: Dict[str, np.ndarray], filename: str):
        """Save simulation results to HDF5 file with metadata"""
        with h5py.File(filename, 'w') as f:
            # Store arrays
            for key, arr in results.items():
                if key != 'params':
                    f.create_dataset(key, data=arr)

            # Store parameters as attributes
            params_group = f.create_group('params')
            for key, value in results['params'].items():
                params_group.attrs[key] = value

            # Store simulation metadata
            f.attrs['simulation_time'] = self.params.simulation_time
            f.attrs['main_channel_width'] = self.params.main_channel_width
            f.attrs['side_channel_width'] = self.params.side_channel_width
            f.attrs['channel_length'] = self.params.channel_length
            f.attrs['rho_continuous'] = self.params.rho_continuous
            f.attrs['rho_dispersed'] = self.params.rho_dispersed
            f.attrs['mu_continuous'] = self.params.mu_continuous
            f.attrs['surface_tension'] = self.params.surface_tension
            f.attrs['mesh_size'] = self.params.mesh_size
            f.attrs['creation_date'] = time.strftime('%Y-%m-%d %H:%M:%S')



Using device: cpu


In [ ]:
# =============================================================================
# Part 2: Academic-Grade PINN Architecture
# =============================================================================

class FourierFeatureTransform(nn.Module):
    """
    Fourier feature mapping for coordinate-based neural networks.
    Helps with learning high-frequency functions.

    Reference: Tancik et al. "Fourier Features Let Networks Learn High Frequency
    Functions in Low Dimensional Domains" (NeurIPS 2020)
    """
    def __init__(self, input_dim: int, num_features: int, sigma: float = 10.0):
        super().__init__()
        self.input_dim = input_dim
        self.num_features = num_features

        # Random Fourier feature matrix
        self.B = nn.Parameter(sigma * torch.randn(input_dim, num_features),
                             requires_grad=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Apply Fourier feature transform to input coordinates

        Args:
            x: Input tensor of shape (..., input_dim)

        Returns:
            Transformed features of shape (..., 2*num_features)
        """
        if x.dim() == 1:
            x = x.unsqueeze(0)

        # Project input onto Fourier basis
        proj = 2 * np.pi * x @ self.B

        # Return concatenated sine and cosine features
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)

class PhysicsInformedNN(nn.Module):
    """
    Physics-Informed Neural Network for microfluidic droplet dynamics

    Architecture:
    1. Input: (x, y, t, μ, Q) → Fourier features
    2. Deep neural network with skip connections
    3. Output: (u, v, p, φ)

    The network is trained with a composite loss function that includes:
    - Data loss (supervised)
    - Physics loss (Navier-Stokes residuals)
    - Boundary condition loss
    - Interface transport loss
    """

    def __init__(self, input_dim: int = 5, hidden_dim: int = 64,
                 num_layers: int = 6, num_features: int = 64):
        super().__init__()

        # Fourier feature embedding
        self.fourier = FourierFeatureTransform(input_dim, num_features)

        # Main network layers
        layers = []
        in_dim = 2 * num_features  # Output dimension of Fourier features

        # Input layer
        layers.append(nn.Linear(in_dim, hidden_dim))
        layers.append(nn.Tanh())

        # Hidden layers with skip connections
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.Tanh())

        # Output layer
        layers.append(nn.Linear(hidden_dim, 4))  # u, v, p, φ

        self.net = nn.Sequential(*layers)

        # Initialize weights
        self._initialize_weights()

    def _initialize_weights(self):
        """Initialize network weights using Xavier/Glorot initialization"""
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_normal_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the network

        Args:
            x: Input tensor of shape (batch_size, input_dim)
               where input_dim = (x, y, t, μ, Q)

        Returns:
            Tensor of shape (batch_size, 4) containing (u, v, p, φ)
        """
        # Apply Fourier feature transform
        features = self.fourier(x)

        # Pass through main network
        output = self.net(features)

        return output

    def compute_losses(self, x: torch.Tensor, y: Optional[torch.Tensor] = None,
                       params: Optional[Dict] = None) -> Dict[str, torch.Tensor]:
        """
        Compute all loss terms for the PINN

        Args:
            x: Input coordinates (batch_size, input_dim)
            y: Ground truth values (batch_size, 4) or None
            params: Dictionary of physical parameters

        Returns:
            Dictionary of loss terms
        """
        losses = {}

        # Ensure we're in training mode for gradient computation
        self.train()

        # Require gradients for physics loss computation
        x.requires_grad_(True)

        # Network prediction
        pred = self(x)
        u_pred = pred[:, 0:1]
        v_pred = pred[:, 1:2]
        p_pred = pred[:, 2:3]
        phi_pred = pred[:, 3:4]

        # Compute gradients for physics loss
        du_dx = torch.autograd.grad(u_pred, x, grad_outputs=torch.ones_like(u_pred),
                                   create_graph=True)[0][:, 0:1]
        du_dy = torch.autograd.grad(u_pred, x, grad_outputs=torch.ones_like(u_pred),
                                   create_graph=True)[0][:, 1:2]
        du_dt = torch.autograd.grad(u_pred, x, grad_outputs=torch.ones_like(u_pred),
                                   create_graph=True)[0][:, 2:3]

        dv_dx = torch.autograd.grad(v_pred, x, grad_outputs=torch.ones_like(v_pred),
                                   create_graph=True)[0][:, 0:1]
        dv_dy = torch.autograd.grad(v_pred, x, grad_outputs=torch.ones_like(v_pred),
                                   create_graph=True)[0][:, 1:2]
        dv_dt = torch.autograd.grad(v_pred, x, grad_outputs=torch.ones_like(v_pred),
                                   create_graph=True)[0][:, 2:3]

        dp_dx = torch.autograd.grad(p_pred, x, grad_outputs=torch.ones_like(p_pred),
                                   create_graph=True)[0][:, 0:1]
        dp_dy = torch.autograd.grad(p_pred, x, grad_outputs=torch.ones_like(p_pred),
                                   create_graph=True)[0][:, 1:2]

        dphi_dx = torch.autograd.grad(phi_pred, x, grad_outputs=torch.ones_like(phi_pred),
                                     create_graph=True)[0][:, 0:1]
        dphi_dy = torch.autograd.grad(phi_pred, x, grad_outputs=torch.ones_like(phi_pred),
                                     create_graph=True)[0][:, 1:2]
        dphi_dt = torch.autograd.grad(phi_pred, x, grad_outputs=torch.ones_like(phi_pred),
                                     create_graph=True)[0][:, 2:3]

        # Second derivatives for viscous terms
        d2u_dx2 = torch.autograd.grad(du_dx, x, grad_outputs=torch.ones_like(du_dx),
                                     create_graph=True)[0][:, 0:1]
        d2u_dy2 = torch.autograd.grad(du_dy, x, grad_outputs=torch.ones_like(du_dy),
                                     create_graph=True)[0][:, 1:2]

        d2v_dx2 = torch.autograd.grad(dv_dx, x, grad_outputs=torch.ones_like(dv_dx),
                                     create_graph=True)[0][:, 0:1]
        d2v_dy2 = torch.autograd.grad(dv_dy, x, grad_outputs=torch.ones_like(dv_dy),
                                     create_graph=True)[0][:, 1:2]

        # Compute density and viscosity based on phase field
        rho = params['rho_continuous'] * (phi_pred > 0).float() + \
              params['rho_dispersed'] * (phi_pred <= 0).float()
        mu = params['mu_continuous'] * (phi_pred > 0).float() + \
             x[:, 3:4] * (phi_pred <= 0).float()  # μ_dispersed is an input

        # Surface tension force (simplified)
        grad_phi_mag = torch.sqrt(dphi_dx**2 + dphi_dy**2 + 1e-10)
        kappa = (dphi_dx**2 * torch.autograd.grad(dphi_dy, x,
                                                grad_outputs=torch.ones_like(dphi_dy),
                                                create_graph=True)[0][:, 0:1] -
                2 * dphi_dx * dphi_dy * torch.autograd.grad(dphi_dx, x,
                                                           grad_outputs=torch.ones_like(dphi_dx),
                                                           create_graph=True)[0][:, 1:2] +
                dphi_dy**2 * torch.autograd.grad(dphi_dx, x,
                                               grad_outputs=torch.ones_like(dphi_dx),
                                               create_graph=True)[0][:, 0:1]) / (grad_phi_mag**3 + 1e-10)

        F_st_x = params['surface_tension'] * kappa * dphi_dx
        F_st_y = params['surface_tension'] * kappa * dphi_dy

        # 1. Data loss (if ground truth is provided)
        if y is not None:
            losses['data_u'] = torch.mean((u_pred - y[:, 0:1])**2)
            losses['data_v'] = torch.mean((v_pred - y[:, 1:2])**2)
            losses['data_p'] = torch.mean((p_pred - y[:, 2:3])**2)
            losses['data_phi'] = torch.mean((phi_pred - y[:, 3:4])**2)

        # 2. Physics loss (Navier-Stokes equations)
        # Continuity equation
        losses['continuity'] = torch.mean((du_dx + dv_dy)**2)

        # x-momentum
        momentum_x = (rho * (du_dt + u_pred * du_dx + v_pred * du_dy) +
                     dp_dx -
                     mu * (d2u_dx2 + d2u_dy2) -
                     F_st_x)
        losses['momentum_x'] = torch.mean(momentum_x**2)

        # y-momentum
        momentum_y = (rho * (dv_dt + u_pred * dv_dx + v_pred * dv_dy) +
                     dp_dy -
                     mu * (d2v_dx2 + d2v_dy2) -
                     F_st_y)
        losses['momentum_y'] = torch.mean(momentum_y**2)

        # 3. Interface transport loss
        interface_transport = dphi_dt + u_pred * dphi_dx + v_pred * dphi_dy
        losses['interface'] = torch.mean(interface_transport**2)

        # 4. Boundary condition losses (simplified examples)
        # Identify boundary points (would need proper boundary masks in practice)
        # Left boundary (inlet)
        left_boundary = (x[:, 0] < 0.1 * params['channel_length'])
        if torch.any(left_boundary):
            # No-slip condition for walls
            wall_mask = (x[:, 1] < 0.1 * params['main_channel_width']) | \
                        (x[:, 1] > 0.9 * params['main_channel_width'])
            wall_points = left_boundary & wall_mask
            if torch.any(wall_points):
                losses['bc_wall_u'] = torch.mean(u_pred[wall_points]**2)
                losses['bc_wall_v'] = torch.mean(v_pred[wall_points]**2)

            # Inlet velocity condition
            inlet_mask = ~wall_mask & left_boundary
            if torch.any(inlet_mask):
                # Simplified: assume constant inlet velocity
                target_u = torch.where(x[inlet_mask, 1] < 0.5 * params['main_channel_width'],
                                     x[inlet_mask, 4:5],  # Q (flow rate ratio)
                                     params['u_continuous'] * torch.ones_like(u_pred[inlet_mask]))
                losses['bc_inlet_u'] = torch.mean((u_pred[inlet_mask] - target_u)**2)
                losses['bc_inlet_v'] = torch.mean(v_pred[inlet_mask]**2)

        # Right boundary (outlet)
        right_boundary = (x[:, 0] > 0.9 * params['channel_length'])
        if torch.any(right_boundary):
            # Pressure outlet condition
            losses['bc_outlet_p'] = torch.mean(p_pred[right_boundary]**2)

        return losses

class AdaptiveWeightedLoss(nn.Module):
    """
    Adaptive loss weighting scheme for PINNs

    Implements the approach from:
    "Adaptive Physics-Informed Neural Networks for Markov-Chain Monte Carlo" (2021)
    """
    def __init__(self, num_loss_terms: int, initial_weights: Optional[List[float]] = None):
        super().__init__()
        self.num_loss_terms = num_loss_terms

        # Initialize learnable loss weights
        if initial_weights is None:
            initial_weights = [1.0] * num_loss_terms

        self.log_weights = nn.Parameter(torch.log(torch.tensor(initial_weights)),
                                      requires_grad=True)

    def forward(self, losses: Dict[str, torch.Tensor]) -> torch.Tensor:
        """
        Compute weighted loss from individual loss terms

        Args:
            losses: Dictionary of loss terms

        Returns:
            Weighted total loss
        """
        # Convert dictionary to ordered list
        loss_terms = list(losses.values())

        # Ensure we have the right number of terms
        if len(loss_terms) != self.num_loss_terms:
            raise ValueError(f"Expected {self.num_loss_terms} loss terms, got {len(loss_terms)}")

        # Compute weighted loss
        total_loss = 0.0
        for i, loss in enumerate(loss_terms):
            weight = torch.exp(-self.log_weights[i])
            total_loss += weight * loss + 0.5 * self.log_weights[i]  # Regularization

        return total_loss

    def get_weights(self) -> List[float]:
        """Get current loss weights"""
        return torch.exp(-self.log_weights.detach()).tolist()

class PINNTrainer:
    """
    Training manager for Physics-Informed Neural Networks

    Handles:
    - Data loading and batching
    - Training loop with adaptive weighting
    - Learning rate scheduling
    - Model checkpointing
    - Progress monitoring
    """

    def __init__(self, model: nn.Module, params: Dict,
                 train_data: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
                 collocation_points: Optional[torch.Tensor] = None):
        self.model = model.to(device)
        self.params = params

        # Set up data
        self.train_data = train_data
        self.collocation_points = collocation_points

        # Set up optimizer
        self.optimizer = optim.Adam([
            {'params': self.model.parameters()},
            {'params': self.model.fourier.parameters(), 'lr': params.get('fourier_lr', 1e-3)}
        ], lr=params.get('lr', 1e-3))

        # Learning rate scheduler
        self.scheduler = ReduceLROnPlateau(self.optimizer, mode='min',
                                          factor=0.5, patience=100, verbose=True)

        # Adaptive loss weighting
        self.loss_weights = AdaptiveWeightedLoss(num_loss_terms=9)  # Adjust based on actual loss terms

        # Training history
        self.history = {
            'total_loss': [],
            'data_loss': [],
            'physics_loss': [],
            'lr': []
        }

        # Create output directory
        os.makedirs(params.get('output_dir', 'results'), exist_ok=True)

    def train(self, num_epochs: int, batch_size: int = 1024):
        """Main training loop"""
        start_time = time.time()

        # Create data loaders if data is provided
        if self.train_data is not None:
            train_dataset = TensorDataset(*self.train_data)
            train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        else:
            train_loader = None

        if self.collocation_points is not None:
            colloc_loader = DataLoader(self.collocation_points, batch_size=batch_size, shuffle=True)
        else:
            colloc_loader = None

        # Training loop
        for epoch in range(num_epochs):
            self.model.train()
            epoch_loss = 0.0
            epoch_data_loss = 0.0
            epoch_physics_loss = 0.0
            num_batches = 0

            # Mixed batch training (data + collocation points)
            if train_loader is not None and colloc_loader is not None:
                data_iter = iter(train_loader)
                colloc_iter = iter(colloc_loader)

                try:
                    while True:
                        # Get a batch of data points
                        try:
                            x_data, y_data = next(data_iter)
                            x_data, y_data = x_data.to(device), y_data.to(device)
                        except StopIteration:
                            data_iter = iter(train_loader)
                            x_data, y_data = next(data_iter)
                            x_data, y_data = x_data.to(device), y_data.to(device)

                        # Get a batch of collocation points
                        try:
                            x_colloc = next(colloc_iter)
                            x_colloc = x_colloc.to(device)
                        except StopIteration:
                            colloc_iter = iter(colloc_loader)
                            x_colloc = next(colloc_iter)
                            x_colloc = x_colloc.to(device)

                        # Zero gradients
                        self.optimizer.zero_grad()

                        # Compute losses for data points
                        losses_data = self.model.compute_losses(x_data, y_data, self.params)

                        # Compute losses for collocation points
                        losses_colloc = self.model.compute_losses(x_colloc, None, self.params)

                        # Combine losses
                        combined_losses = {}
                        for key in losses_data:
                            combined_losses[key] = losses_data[key] + losses_colloc.get(key, 0.0)

                        # Compute weighted loss
                        total_loss = self.loss_weights(combined_losses)

                        # Backpropagate
                        total_loss.backward()

                        # Gradient clipping
                        torch.nn.utils.clip_g